# STT — 02 · Prétraitement

On suit **un seul clip** pas à pas : signal brut → signal à 16 kHz → log-Mel → texte encodé, en regardant à chaque étape la forme du tenseur.

## Étape 1 — Lire un clip et le convertir en tenseur

In [ ]:
import pandas as pd
import soundfile as sf
import torch

# On prend le premier clip d'entraînement de fr-FR
train_fr = pd.read_csv("data/fr-FR/train.tsv", sep="	")
clip = train_fr.iloc[0]
print("Fichier :", clip["path"])
print("Phrase :", clip["sentence"])

# Lecture : soundfile renvoie un tableau NumPy et la fréquence d'échantillonnage
audio, sr = sf.read(f"data/fr-FR/clips/{clip['path']}", dtype="float32")
print("Type :", type(audio), "| forme :", audio.shape, "| fréquence :", sr, "Hz")

# Conversion en tenseur PyTorch (partage la même mémoire, sans copie)
waveform = torch.from_numpy(audio)
print("Tenseur :", waveform.shape, waveform.dtype)
print("Durée :", len(waveform) / sr, "s")
print("Amplitude min / max :", waveform.min().item(), waveform.max().item())

## Étape 2 — Rééchantillonner à 16 kHz

`resample` applique d'abord un **filtre passe-bas** (anti-aliasing : supprime ce qui dépasse 8 kHz, la limite de Nyquist à 16 kHz), puis réduit le nombre d'échantillons.

In [2]:
import torchaudio.functional as AF

TARGET_SR = 16000  # décision de l'EDA, point 4

# Filtre passe-bas (anti-aliasing) puis réduction du nombre d'échantillons
waveform_16k = AF.resample(waveform, orig_freq=sr, new_freq=TARGET_SR)

print("Avant :", waveform.shape, "à", sr, "Hz")
print("Après :", waveform_16k.shape, "à", TARGET_SR, "Hz")
print("Rapport :", len(waveform) / len(waveform_16k))
print("Durée après :", len(waveform_16k) / TARGET_SR, "s")

Avant : torch.Size([223488]) à 48000 Hz
Après : torch.Size([74496]) à 16000 Hz
Rapport : 3.0
Durée après : 4.656 s


## Étape 3 — Spectrogramme log-Mel

**Décision** : fenêtre 25 ms (400 échantillons), pas 10 ms (160 échantillons), 80 bandes Mel — standard de la reconnaissance vocale, identique à Whisper (comparaison plus juste).

In [3]:
import torchaudio.transforms as T

N_FFT = 400        # taille de fenêtre : 400 échantillons = 25 ms à 16 kHz
HOP_LENGTH = 160   # pas : 160 échantillons = 10 ms -> 100 instants par seconde
N_MELS = 80        # nombre de bandes Mel

# Objet qui enchaîne STFT puis regroupement en bandes Mel
mel_transform = T.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
)

mel = mel_transform(waveform_16k)
print("Spectrogramme Mel :", mel.shape)   # (bandes, instants)

# Logarithme ; le petit 1e-6 évite log(0) = -infini sur les silences parfaits
log_mel = torch.log(mel + 1e-6)

# On transpose pour avoir (instants, caractéristiques), comme dans un Transformer
features = log_mel.T
print("Entrée du modèle :", features.shape)
print("Min / max / moyenne / écart-type :",
      features.min().item(), features.max().item(),
      features.mean().item(), features.std().item())

Spectrogramme Mel : torch.Size([80, 466])
Entrée du modèle : torch.Size([466, 80])
Min / max / moyenne / écart-type : -13.815510749816895 6.605799198150635 -7.790489196777344 5.582396507263184


## Étape 4 — Normalisation par clip

**Décision** : normalisation **par clip** (moyenne 0, écart-type 1, bande par bande) — compense les différences de micro et de volume, utile pour le passage Common Voice → robot.

In [4]:
# Moyenne et écart-type de chaque bande Mel, calculés sur tous les instants de ce clip
mean = features.mean(dim=0, keepdim=True)   # forme (1, 80)
std = features.std(dim=0, keepdim=True)     # forme (1, 80)

# Centrer (moyenne 0) et réduire (écart-type 1), bande par bande.
# Le 1e-5 évite une division par 0 si une bande est constante dans tout le clip.
features_norm = (features - mean) / (std + 1e-5)

print("Forme de la moyenne :", mean.shape)
print("Forme après normalisation :", features_norm.shape)
print("Moyenne globale :", features_norm.mean().item())
print("Écart-type global :", features_norm.std().item())

Forme de la moyenne : torch.Size([1, 80])
Forme après normalisation : torch.Size([466, 80])
Moyenne globale : -2.7832555815621163e-08
Écart-type global : 0.9989376664161682


## Étape 5 — Encodage du texte

Chaque caractère reçoit un indice ; le **blank** prend l'indice **0** (valeur par défaut de `nn.CTCLoss`).

In [ ]:
import re
import unicodedata

# Mêmes règles que celles validées au point 2 de l'EDA
ALLOWED = set("abcdefghijklmnopqrstuvwxyz" "àâæçéèêëîïôœùûüÿ" " '-.,?!")

def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"[’‘ʼ´]", "'", text)
    text = re.sub(r"[;:…«»()\"—–]", " ", text)
    text = re.sub(r" +", " ", text).strip()
    return text

# Vocabulaire : le blank en position 0, puis les 49 caractères triés
BLANK = "_"
vocab = [BLANK] + sorted(ALLOWED)

# Deux dictionnaires : caractère -> indice (encoder) et indice -> caractère (décoder)
char_to_id = {char: i for i, char in enumerate(vocab)}
id_to_char = {i: char for i, char in enumerate(vocab)}

text = normalize(clip["sentence"])
target = torch.tensor([char_to_id[char] for char in text])

print("Taille du vocabulaire :", len(vocab))
print("Texte normalisé :", text)
print("Indices :", target)
print("Forme :", target.shape)

# Vérification : décoder les indices doit redonner exactement le texte
decoded = "".join(id_to_char[i] for i in target.tolist())
print("Décodage identique :", decoded == text)

## Vérification du module `stt/`

Le code validé est rangé dans `stt/text.py` et `stt/audio.py`. On vérifie qu'il donne exactement le même résultat que les cellules ci-dessus.

In [ ]:
from stt.audio import load_audio, log_mel
from stt.text import normalize, encode, decode

features_module = log_mel(load_audio(f"data/fr-FR/clips/{clip['path']}"))
target_module = encode(normalize(clip["sentence"]))

print("Forme :", features_module.shape)
# allclose : égalité à d'infimes erreurs d'arrondi près
print("Features identiques :", torch.allclose(features_module, features_norm, atol=1e-5))
print("Cible identique :", torch.equal(target_module, target))
print("Décodage :", decode(target_module.tolist()))

## Étape 6 — Le `Dataset` PyTorch

`stt/dataset.py` : `load_split` assemble les tableaux d'un découpage (fr-FR + fr-CA), `CommonVoiceDataset` renvoie le clip n° *i* prétraité à la demande.

In [7]:
from stt.dataset import load_split, CommonVoiceDataset

folders = ["data/fr-FR", "data/fr-CA"]
train_table = load_split(folders, "train")
dev_table = load_split(folders, "dev")

train_dataset = CommonVoiceDataset(train_table)
dev_dataset = CommonVoiceDataset(dev_table)
print("Clips train :", len(train_dataset))
print("Clips dev :", len(dev_dataset))

# On demande un clip au hasard, comme le fera l'entraînement
features, target = train_dataset[100]
print("Features :", features.shape, "| cible :", target.shape)

Clips train : 12982
Clips dev : 484
Features : torch.Size([495, 80]) | cible : torch.Size([64])


## Étape 7 — DataLoader et padding

`collate_batch` complète les clips d'un batch avec des 0 jusqu'à la longueur du plus long, et renvoie les vraies longueurs (pour le CTC et le masque d'attention).

In [8]:
from torch.utils.data import DataLoader
from stt.dataset import collate_batch

train_loader = DataLoader(
    train_dataset,
    batch_size=8,              # 8 clips par batch
    shuffle=True,              # ordre aléatoire à chaque époque
    collate_fn=collate_batch,  # notre fonction de padding
)

# On prend un seul batch pour regarder les formes
features, feature_lengths, targets, target_lengths = next(iter(train_loader))
print("Features :", features.shape)
print("Longueurs réelles :", feature_lengths)
print("Cibles :", targets.shape)
print("Longueurs des phrases :", target_lengths)

Features : torch.Size([8, 956, 80])
Longueurs réelles : tensor([956, 695, 629, 353, 349, 339, 652, 392])
Cibles : torch.Size([8, 99])
Longueurs des phrases : tensor([99, 73, 89, 36, 53, 30, 80, 34])


## Premier bloc du modèle — Sous-échantillonnage par convolutions 2D

`stt/model.py` : `ConvSubsampling` divise le nombre d'instants par ~4. Valeurs **provisoires** pour le test (`channels=32`, `d_model=144`) : la taille du modèle sera décidée ensuite.

In [9]:
from stt.model import ConvSubsampling

# Valeurs provisoires, uniquement pour vérifier les formes
subsampling = ConvSubsampling(n_mels=80, channels=32, d_model=144)

# Forward pass sur le batch de l'étape 7
out, out_lengths = subsampling(features, feature_lengths)

print("Entrée :", features.shape, "| longueurs :", feature_lengths)
print("Sortie :", out.shape, "| longueurs :", out_lengths)

Entrée : torch.Size([8, 956, 80]) | longueurs : tensor([956, 695, 629, 353, 349, 339, 652, 392])
Sortie : torch.Size([8, 238, 144]) | longueurs : tensor([238, 173, 156,  87,  86,  84, 162,  97])


## Bloc 2 du modèle — Encodage sinusoïdal des positions

**Décision** : encodage sinusoïdal (Transformer original). Passage à un encodage relatif/rotatif si le WER se dégrade fortement sur les clips longs.

In [10]:
from stt.model import SinusoidalPositionalEncoding

pos_enc = SinusoidalPositionalEncoding(d_model=144)
out_pos = pos_enc(out)   # « out » = sortie du sous-échantillonnage

print("Forme avant / après :", out.shape, out_pos.shape)
print("Paramètres appris :", sum(p.numel() for p in pos_enc.parameters()))
print("Position 0, 4 premières valeurs :", pos_enc.pe[0, :4])
print("Position 1, 4 premières valeurs :", pos_enc.pe[1, :4])

Forme avant / après : torch.Size([8, 238, 144]) torch.Size([8, 238, 144])
Paramètres appris : 0
Position 0, 4 premières valeurs : tensor([0., 1., 0., 1.])
Position 1, 4 premières valeurs : tensor([0.8415, 0.5403, 0.7707, 0.6372])


## Bloc 3, morceau 1 — Attention à une tête, avec masque du padding

**Décision (option C)** : on écrit l'attention à la main pour la comprendre, on la **vérifie contre PyTorch** (test de référence), et l'entraînement utilisera la version optimisée de PyTorch.

In [11]:
import torch.nn.functional as F
from stt.model import make_padding_mask, scaled_dot_product_attention

torch.manual_seed(0)   # pour avoir toujours les mêmes nombres aléatoires

# 2 séquences de 4 instants, d_k = 8 ; la 2e n'a que 2 vrais instants (le reste = padding)
lengths = torch.tensor([4, 2])
mask = make_padding_mask(lengths, max_len=4)
q = torch.randn(2, 4, 8)
k = torch.randn(2, 4, 8)
v = torch.randn(2, 4, 8)

output, weights = scaled_dot_product_attention(q, k, v, mask)

print("Masque :\n", mask)
print("Sortie :", output.shape, "| poids :", weights.shape)
print("Poids de la 2e séquence (lignes = qui regarde, colonnes = qui est regardé) :\n", weights[1].round(decimals=2))
print("Somme de chaque ligne :", weights.sum(dim=-1))

# Test de référence : version optimisée de PyTorch, même convention de masque
# (True = autorisé à regarder), forme (B, 1, T) pour masquer les colonnes.
output_pytorch = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, :])
print("Identique à PyTorch :", torch.allclose(output, output_pytorch, atol=1e-6))

Masque :
 tensor([[ True,  True,  True,  True],
        [ True,  True, False, False]])
Sortie : torch.Size([2, 4, 8]) | poids : torch.Size([2, 4, 4])
Poids de la 2e séquence (lignes = qui regarde, colonnes = qui est regardé) :
 tensor([[0.3000, 0.7000, 0.0000, 0.0000],
        [0.1400, 0.8600, 0.0000, 0.0000],
        [0.3200, 0.6800, 0.0000, 0.0000],
        [0.6500, 0.3500, 0.0000, 0.0000]])
Somme de chaque ligne : tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])
Identique à PyTorch : True


## Bloc 3, morceau 2 — Multi-head attention

Test de référence contre `nn.MultiheadAttention` (mêmes poids copiés), puis contre la version optimisée.

In [12]:
from torch import nn
from stt.model import MultiHeadSelfAttention

torch.manual_seed(0)
x = torch.randn(2, 4, 144)        # 2 séquences, 4 instants, d_model = 144
lengths = torch.tensor([4, 2])
mask = make_padding_mask(lengths, max_len=4)

ours = MultiHeadSelfAttention(d_model=144, n_heads=4, use_pytorch_kernel=False)

# Module de PyTorch, auquel on copie EXACTEMENT nos poids pour comparer à l'identique
reference = nn.MultiheadAttention(embed_dim=144, num_heads=4, batch_first=True)
with torch.no_grad():
    reference.in_proj_weight.copy_(ours.qkv.weight)
    reference.in_proj_bias.copy_(ours.qkv.bias)
    reference.out_proj.weight.copy_(ours.out.weight)
    reference.out_proj.bias.copy_(ours.out.bias)

out_ours = ours(x, mask)
# Convention PyTorch inversée : key_padding_mask = True là où il faut IGNORER
out_ref, _ = reference(x, x, x, key_padding_mask=~mask)

# On compare les vrais instants uniquement
print("Sortie :", out_ours.shape)
print("Identique à nn.MultiheadAttention :", torch.allclose(out_ours[mask], out_ref[mask], atol=1e-5))

# La version optimisée doit aussi donner le même résultat
ours.use_pytorch_kernel = True
print("Version optimisée identique :", torch.allclose(ours(x, mask)[mask], out_ours[mask], atol=1e-5))

Sortie : torch.Size([2, 4, 144])
Identique à nn.MultiheadAttention : True
Version optimisée identique : True


## Bloc 3, morceaux 3 et 4 — Feed-forward et bloc Transformer complet

**Décisions** : pre-norm, GELU, dropout 0,1. Test de référence contre `nn.TransformerEncoderLayer` (`norm_first=True`, `activation="gelu"`).

In [13]:
from stt.model import TransformerBlock

torch.manual_seed(0)
x = torch.randn(2, 4, 144)
mask = make_padding_mask(torch.tensor([4, 2]), max_len=4)

ours = TransformerBlock(d_model=144, n_heads=4, d_ff=576, dropout=0.1, use_pytorch_kernel=False)

# Bloc de PyTorch configuré comme le nôtre : GELU, pre-norm (norm_first=True)
reference = nn.TransformerEncoderLayer(
    d_model=144, nhead=4, dim_feedforward=576, dropout=0.1,
    activation="gelu", batch_first=True, norm_first=True,
)

# Copie de nos poids dans le bloc de PyTorch
with torch.no_grad():
    reference.self_attn.in_proj_weight.copy_(ours.attention.qkv.weight)
    reference.self_attn.in_proj_bias.copy_(ours.attention.qkv.bias)
    reference.self_attn.out_proj.weight.copy_(ours.attention.out.weight)
    reference.self_attn.out_proj.bias.copy_(ours.attention.out.bias)
    reference.linear1.weight.copy_(ours.feed_forward.net[0].weight)
    reference.linear1.bias.copy_(ours.feed_forward.net[0].bias)
    reference.linear2.weight.copy_(ours.feed_forward.net[3].weight)
    reference.linear2.bias.copy_(ours.feed_forward.net[3].bias)
    reference.norm1.load_state_dict(ours.norm1.state_dict())
    reference.norm2.load_state_dict(ours.norm2.state_dict())

# Mode évaluation : désactive le dropout (sinon les zéros aléatoires différeraient)
ours.eval()
reference.eval()

out_ours = ours(x, mask)
out_ref = reference(x, src_key_padding_mask=~mask)
print("Sortie :", out_ours.shape)
print("Identique à nn.TransformerEncoderLayer :", torch.allclose(out_ours[mask], out_ref[mask], atol=1e-5))

Sortie : torch.Size([2, 4, 144])
Identique à nn.TransformerEncoderLayer : True


## Le modèle complet — `SpeechTransformer` (taille S)

Tout le pipeline sur un vrai batch : DataLoader -> modèle -> log-probabilités.

In [14]:
from stt.model import SpeechTransformer

model = SpeechTransformer()   # valeurs par défaut = taille S
n_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {n_params / 1e6:.2f} M")

# Un vrai batch : features (B, T, 80) + longueurs, et les cibles
features, feature_lengths, targets, target_lengths = next(iter(train_loader))

model.eval()                  # pas de dropout pour ce test
with torch.no_grad():         # pas de gradients : on ne fait que regarder
    log_probs, out_lengths = model(features, feature_lengths)

print("Entrée :", features.shape)
print("Sortie :", log_probs.shape)
print("Instants disponibles :", out_lengths)
print("Caractères à écrire  :", target_lengths)
print("Contrainte CTC respectée :", bool((out_lengths >= target_lengths).all()))
# exp(log-probabilités) = probabilités : elles doivent sommer à 1 pour chaque instant
print("Probabilités sommant à 1 :", torch.allclose(log_probs.exp().sum(dim=-1), torch.ones(1)))

Paramètres : 3.60 M


Entrée : torch.Size([8, 627, 80])
Sortie : torch.Size([8, 156, 50])
Instants disponibles : tensor([ 94, 156, 108, 135, 126,  93, 139, 152])
Caractères à écrire  : tensor([55, 76, 59, 52, 70, 35, 65, 72])
Contrainte CTC respectée : True
Probabilités sommant à 1 : True
